# Ramsey spectroscopy for photon detection

> Simulation of the qubit-based single-photon counter of **Dixit *et al.*, PRL 126, 141302 (2021)** — a dark matter search with a superconducting qubit.

A transmon qubit is dispersively coupled to a high-$Q$ **storage cavity**, where dark matter (axion / hidden photon) may on rare occasions deposit a single photon. The photon is never absorbed by the detector: its presence only shifts the qubit frequency by $2\chi$ per photon, and a **Ramsey sequence** converts that frequency shift into a qubit population that can be read out — repeatedly and non-destructively.

**Ramsey = an interferometer made of two $\pi/2$ pulses with a waiting time in between:**

| step | duration | what it does |
| :--- | :--- | :--- |
| $\pi/2$ pulse | $t_{\pi/2} = \pi/2A$ | puts the qubit on the equator, $\tfrac{1}{\sqrt{2}}(\lvert g\rangle - i\lvert e\rangle)$ |
| free evolution | $\tau$ | the two branches accumulate a **relative phase** $\varphi = 2\chi n\,\tau$, with $n$ the cavity photon number |
| $\pi/2$ pulse | $t_{\pi/2}$ | folds that phase back into a $\lvert g\rangle / \lvert e\rangle$ **population**, which is what a measurement can actually see |

A phase is invisible to a measurement; a population is not. The two $\pi/2$ pulses are just the machinery that turns one into the other — that is the whole point of the technique.

Picking $\tau_\pi = \pi/2\chi$ gives $\varphi = \pi$ for one photon and $\varphi = 0$ for none, so the sequence acts as a qubit $\pi$ pulse **conditioned on the presence of a cavity photon** (a photon-number *parity* measurement, in the paper's language).

---

The notebook runs that sequence twice:

1. **Rotating frame (RWA)** — sections 1 to 3, one section per pulse, each with the analytic evolution operator next to the numerics, and the Bloch-sphere picture at the end.
2. **Lab frame, no approximation** — final section. The same sequence driven by the real oscillating field $A\cos(\omega_d t)\,\sigma_x$, which is what the hardware actually delivers.

In [ ]:
import qutip as qt
import matplotlib.pyplot as plt
import numpy as np


## Setting

In the dispersive regime the coupling $g$ is much smaller than the qubit–cavity detuning $\Delta_{qs} = \omega_s - \omega_q$, so the two never exchange energy. The only surviving effect is a shift of the qubit frequency, per photon, of

$$\chi = \frac{g^2}{\Delta_{qs}} \qquad (\texttt{chi\_01})$$

In the frame rotating at the drive frequency, the driven qubit is then

$$H = \frac{1}{2}\left(\omega_q + 2\chi n - \omega_d\right)\sigma_z + \frac{A}{2}\,\sigma_x$$

or, written as a **Rabi Hamiltonian**,

$$\mathcal{H} = \frac{1}{2}\,\Omega\,\hat{n} \cdot \vec{\sigma}$$

$$\Omega = \sqrt{A^2 + (\Delta + 2\chi n)^2}, \qquad \hat{n} = \frac{\left(A,\; 0,\; \Delta + 2\chi n\right)}{\Omega}, \qquad \Delta = \omega_q - \omega_d$$

This is the same structure as a spin in a magnetic field: the Bloch vector precesses about the effective field $\hat{n}$ at rate $\Omega$. The one difference is that the $z$ component of that field **depends on the cavity photon number $n$** — the qubit precesses at a different rate depending on whether a photon is there or not. Everything below is a consequence of that single sentence.

> ⚠️ **Two different detunings share the symbol $\Delta$ in the literature.** Here $\Delta_{qs} = \omega_s - \omega_q$ is the qubit–cavity detuning, which sets $\chi$; while $\Delta = \omega_q - \omega_d$ is the drive detuning, which sets the rotation axis. From here on, $\Delta$ always means the drive detuning.

**Three things to keep in mind while reading the code:**

- The two inline comments are swapped: `omega_s` (6.0 GHz) is the **storage cavity** and `omega_q` (4.5 GHz) is the **qubit** — matching the paper's 6.011 GHz and 4.749 GHz. Only their difference enters $\chi$, so the physics is unaffected.
- The cavity is never actually part of the Hilbert space (`a` is defined but unused). The $n = 0$ and $n = 1$ cases are simulated by hand as two different *qubit* Hamiltonians, `H_0` and `H_1`. That is legitimate precisely because $n$ is conserved in the dispersive regime.
- Frequencies are given as $2\pi \times$ GHz, so all times come out in **ns** (the `s` in some labels should read `ns`).

In [ ]:
# hamiltonian of qubit with a resonator driven

omega_s = 6.0 * 2 * np.pi  # qubit frequency
omega_q = 4.5 * 2 * np.pi  # resonator frequency
omega_d = omega_q  # drive frequency

g = 0.1 * 2 * np.pi  # coupling strength

chi_01 = g**2 / (omega_s - omega_q)  # dispersive shift

A = 0.1 * 2 * np.pi  # drive amplitude

N_photons = 5  # number of photons in the resonator

a = qt.destroy(N_photons)  # annihilation operator for the resonator
a_dag = a.dag()  # creation operator for the resonator

# hamiltonian of the system in the dispersive regime with a drive on the qubit
H_d = 0.5 * (omega_q - omega_d) * qt.sigmaz() + A * qt.sigmax() / 2

psi_0 = qt.basis(2, 0)  # initial state of the qubit (ground state)
omega_d = omega_q  # drive frequency

## 1 -  $\frac{\pi}{2}$ rotation

On resonance ($\omega_d = \omega_q$, so $\Delta = 0$) the $\sigma_z$ term vanishes and only $A\sigma_x/2$ survives: the state rotates about $x$ at rate $A$.

**Derivation.** Suppose we start in the ground state $\lvert g\rangle$ and apply a $\pi/2$ pulse. The evolution operator is

$$U = e^{-i\mathcal{H}t} = e^{-i\,\Omega\,(\hat{n}\cdot\vec{\sigma})\,t/2} = \cos\!\left(\frac{\Omega t}{2}\right) I - i\,\sin\!\left(\frac{\Omega t}{2}\right)\left(\hat{n}\cdot\vec{\sigma}\right)$$

where $\theta = \Omega t$ is the rotation angle about $\hat{n}$, and $\hat{n}\cdot\vec{\sigma} = n_x\sigma_x + n_y\sigma_y + n_z\sigma_z$.

On resonance and with an empty cavity ($\Delta = 0$, $n = 0$) we have $\Omega = A$ and $\hat{n} = (1, 0, 0)$, so

$$U = \cos\!\left(\frac{At}{2}\right) I - i\,\sin\!\left(\frac{At}{2}\right)\sigma_x$$

Applying it for a rotation angle $\Omega t = \pi/2$, that is $t_{\pi/2} = \dfrac{\pi}{2A}$:

$$U_{\pi/2} = \cos\!\left(\frac{\pi}{4}\right) I - i\,\sin\!\left(\frac{\pi}{4}\right)\sigma_x = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & -i \\ -i & 1 \end{pmatrix}$$

and the final state is

$$\lvert \psi_f \rangle = U_{\pi/2}\lvert g\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & -i \\ -i & 1 \end{pmatrix}\begin{pmatrix} 1 \\ 0 \end{pmatrix} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 \\ -i \end{pmatrix} = \frac{1}{\sqrt{2}}\left(\lvert g\rangle - i\lvert e\rangle\right)$$

An equal superposition of ground and excited state with a relative phase of $-i$ — on the Bloch sphere, the equator along $-y$. This is the **beam splitter** of the interferometer: the qubit is now equally likely to be found in either state, and from here on all the information lives in the *phase* between the two branches.

> 📌 **Why the pulse itself barely sees the photon.** We took $\Delta = 0$, but a detuning of order MHz would hardly change $U$: as long as $A \gg \lvert\Delta + 2\chi n\rvert$ (here $A/2\chi \approx 7.5$), the effective field $\hat{n}$ still points essentially along $x$ and $\Omega \approx A$. So the $\pi/2$ pulses are nearly blind to the photon — only the free evolution is sensitive to it. That separation of roles is what makes the sequence a clean interferometer.

In [ ]:
t_pi_2 = np.pi / (2 * A)  # time for a pi/2 rotation

t_limits = [0, t_pi_2]  # time limits for the simulation
n_steps = 1000  # number of time steps for the simulation
t_axis = np.linspace(*t_limits, n_steps)  # time axis for the simulation


states_1 = qt.mesolve(H_d, psi_0, t_axis).states

In [ ]:
Pg = qt.expect(qt.projection(2, 0, 0), states_1)  # probability of being in the ground state of the qubit and vacuum of the resonator
Pe = qt.expect(qt.projection(2, 1, 1), states_1)  # probability of being in the excited state of the qubit and vacuum of the resonator


fig, ax = plt.subplots(1, 1, figsize=(12, 8), sharex=True)
ax.plot(t_axis, Pg, label='P(g, 0)', color='blue')
ax.plot(t_axis, Pe, label='P(e, 0)', color='orange')
ax.set_ylabel('Probability')
ax.legend()  

print(f"Time for a pi/2 rotation: {t_pi_2:.3f} s")
print(f"Probability of being in the ground state at t = {t_pi_2:.3f} s: {Pg[-1]:.3f}")
print(f"Probability of being in the excited state at t = {t_pi_2:.3f} s: {Pe[-1]:.3f}")

## 2- free evolution $\tau$

The drive is now off, and this is where the photon actually does something.

**Derivation.** With no drive the Hamiltonian is diagonal,

$$\mathcal{H}_\text{FREE} = \frac{1}{2}\left(\Delta + 2\chi n\right)\sigma_z$$

so the evolution operator is

$$U_\text{FREE} = e^{-i\mathcal{H}_\text{FREE}t} = \cos\!\left(\frac{(\Delta + 2\chi n)t}{2}\right) I - i\,\sin\!\left(\frac{(\Delta + 2\chi n)t}{2}\right)\sigma_z$$

Applying it to the state left by the $\pi/2$ pulse, for a time $t = \tau$:

$$U_\text{FREE}\,\frac{1}{\sqrt{2}}\left(\lvert g\rangle - i\lvert e\rangle\right) = \frac{1}{\sqrt{2}}\begin{pmatrix} e^{-i(\Delta + 2\chi n)\tau/2} & 0 \\ 0 & e^{\,i(\Delta + 2\chi n)\tau/2} \end{pmatrix}\begin{pmatrix} 1 \\ -i \end{pmatrix}$$

$$= \frac{1}{\sqrt{2}}\left(e^{-i(\Delta + 2\chi n)\tau/2}\lvert g\rangle - i\,e^{\,i(\Delta + 2\chi n)\tau/2}\lvert e\rangle\right)$$

Only a **relative phase** is accumulated, which is exactly what we want:

$$\varphi = \left(\Delta + 2\chi n\right)\tau \quad\xrightarrow{\;\Delta\,=\,0\;}\quad \varphi = 2\chi n\,\tau$$

$$\lvert \psi_\text{FREE} \rangle = \frac{1}{\sqrt{2}}\left(e^{-i\chi n\tau}\lvert g\rangle - i\,e^{\,i\chi n\tau}\lvert e\rangle\right)$$

The measurement probabilities do not change at all during $\tau$ — the Bloch vector simply walks around the equator. The photon shifts the qubit frequency by $2\chi$, so *how far it walks* is what tells you the photon is there.

**In the code** the two cases are simulated separately:

- `H_0` — no photon: zero detuning, so the Bloch vector just sits there.
- `H_1` — one photon: the extra $2\chi$ makes it precess about $z$, accumulating $\varphi = 2\chi\tau$.

The scan runs $\tau$ up to $3\tau_\pi$ to see the full fringe. At $\tau_\pi = \pi/2\chi$ we get $\varphi = \pi$: the two cases are exactly antipodal on the equator, which is where the interferometer has maximum contrast — the best possible discrimination between "photon" and "no photon".

In [ ]:
tau_pi = np.pi / (2 * chi_01) 

step = tau_pi / 100
tau_axis = np.arange(0, 3*tau_pi, step)

n_points = 500

states_tau_0 = []
states_tau_1 = []

psi_tau_0 = states_1[-1]  # state after the pi/2 rotation
psi_tau_1 = states_1[-1]  # state after the pi/2 rotation

H_0 = 0.5 * (omega_q - omega_d) * qt.sigmaz()
H_1 = 0.5 * (omega_q - omega_d + 2 * chi_01) * qt.sigmaz()

for i, tau in enumerate(tau_axis):
    states_tau_0.append(qt.mesolve(H_0, psi_tau_0, np.linspace(0, tau, n_points)).states[-1])
    states_tau_1.append(qt.mesolve(H_1, psi_tau_1, np.linspace(0, tau, n_points)).states[-1])

## 3- $\frac{\pi}{2}$ rotation

The second $\pi/2$ pulse closes the interferometer, rotating the equatorial phase into a population difference that the readout resonator can measure.

**Derivation.** The pulse operator is the same as before,

$$U_{\pi/2} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & -i \\ -i & 1 \end{pmatrix}$$

applied now to the state that came out of the free evolution:

$$\lvert \psi_f \rangle = U_{\pi/2}\lvert \psi_\text{FREE}\rangle = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & -i \\ -i & 1 \end{pmatrix}\frac{1}{\sqrt{2}}\begin{pmatrix} e^{-i\chi n\tau} \\ -i\,e^{\,i\chi n\tau} \end{pmatrix}$$

$$\lvert \psi_f \rangle = -i\left[\sin\!\left(\chi n\tau\right)\lvert g\rangle + \cos\!\left(\chi n\tau\right)\lvert e\rangle\right]$$

so that, in terms of the phase $\varphi = 2\chi n\tau$ accumulated during the free evolution,

$$\boxed{\;P(g, n) = \sin^2\!\left(\chi n\tau\right) = \sin^2\!\left(\frac{\varphi}{2}\right)\;}$$

$$\boxed{\;P(e, n) = \cos^2\!\left(\chi n\tau\right) = \cos^2\!\left(\frac{\varphi}{2}\right)\;}$$

**Reading the result.** With one photon the qubit oscillates, $P(g, 1) = \sin^2(\chi\tau)$, at angular frequency $2\chi$. With no photon it is frozen: $P(g, 0) = 0$ and $P(e, 0) = 1$.

To detect the photon we therefore wait exactly $\tau_\pi = \dfrac{\pi}{2\chi}$, which makes $\chi\tau_\pi = \pi/2$:

| cavity | $P(g)$ | $P(e)$ |
| :--- | :---: | :---: |
| **1 photon** | $1$ | $0$ |
| **0 photons** | $0$ | $1$ |

One bit of information about the cavity, carried out on the qubit. That is the whole detector.

In the plot below, read $P(\tau)$ for the two cases: **flat** for $n = 0$ (nothing precessed) and **oscillating** for $n = 1$, with the dashed red line marking $\tau_\pi$.

**Two side notes:**

- Because both pulses here rotate about the *same* axis with the same sign, they compose into a full $\pi$ rotation when $\varphi = 0$. So the mapping comes out inverted relative to the paper: no photon $\rightarrow \lvert e\rangle$, one photon $\rightarrow \lvert g\rangle$. Flipping the sign of the second pulse restores the paper's convention. Nothing physical depends on it.
- In a lab Ramsey experiment one usually detunes the drive deliberately so that *both* branches oscillate, and reads off the frequency difference. Here $\omega_d = \omega_q$ exactly, so the $n = 0$ branch is perfectly flat.

In [ ]:
states_final_0 = []
states_final_1 = []

save_full_0 = []
save_full_1 = []

n_points_final = 500

tau_pi = np.pi / (2 * chi_01) 
tau_pi_2 = tau_pi / 2
tau_pi_4 = tau_pi / 4

tau_full = [tau_pi_4, tau_pi_2, tau_pi]


for i, tau in enumerate(tau_axis):
    
    states_0 = qt.mesolve(H_d, states_tau_0[i], np.linspace(0, t_pi_2, n_points_final)).states
    states_1 = qt.mesolve(H_d, states_tau_1[i], np.linspace(0, t_pi_2, n_points_final)).states
    
    if np.isclose(tau, tau_pi_4) or np.isclose(tau, tau_pi_2) or np.isclose(tau, tau_pi):
        print(f"Saving full states for tau = {tau:.3f} s")
        save_full_0.append(states_0)
        save_full_1.append(states_1)
    
    states_final_0.append(states_0[-1])
    states_final_1.append(states_1[-1])

In [ ]:
Pg0 = qt.expect(qt.projection(2, 0, 0), states_final_0)  # probability of being in the ground state of the qubit and vacuum of the resonator
Pe0 = qt.expect(qt.projection(2, 1, 1), states_final_0)  # probability of being in the excited state of the qubit and vacuum of the resonator
Pg1 = qt.expect(qt.projection(2, 0, 0), states_final_1)  # probability of being in the ground state of the qubit and vacuum of the resonator
Pe1 = qt.expect(qt.projection(2, 1, 1), states_final_1)  # probability of being in the excited state of the qubit and vacuum of the resonator

start, end = 0, 1000

delta_0 = omega_d 

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax[0].plot(tau_axis[start:end], Pg0[start:end], label='P(g, 0)', color='blue')
ax[0].plot(tau_axis[start:end], Pe0[start:end], label='P(e, 0)', color='orange')
ax[0].set_ylabel('Probability')
ax[0].legend()
ax[1].plot(tau_axis[start:end], Pg1[start:end], label='P(g, 1)', color='blue')
ax[1].plot(tau_axis[start:end], Pe1[start:end], label='P(e, 1)', color='orange')
ax[1].axvline(tau_pi, color='red', linestyle='--', label=r'$\tau_\pi$')
ax[1].set_ylabel('Probability')
ax[1].set_xlabel('Free evolution time (ns)')
ax[1].legend()

### Reading the Bloch spheres

Left = no photon, right = one photon, for three values of $\tau$.

Careful with what is being drawn: `save_full_*` only stores the trajectory of the **second $\pi/2$ pulse**, so each trace starts from a point that *already* carries the phase $\varphi = 2\chi\tau$ picked up during the free evolution. The precession around the equator is not animated — it is baked into the starting point of each trace.

$$\tau_\pi/4 \Rightarrow \varphi = \frac{\pi}{4}, \qquad \tau_\pi/2 \Rightarrow \varphi = \frac{\pi}{2}, \qquad \tau_\pi \Rightarrow \varphi = \pi$$

The no-photon trace ends in the same place every time; the one-photon trace ends further and further away from it, reaching the opposite pole exactly at $\tau_\pi$. That growing separation between left and right *is* the detector's signal.

In [ ]:
from IPython.display import display

for i, (states_0, states_1) in enumerate(zip(save_full_0, save_full_1)):
    fig = plt.figure(figsize=(12, 8))
    
    for state, label in zip([states_0, states_1], ['(g, 0)', '(e, 1)']):
        ax = fig.add_subplot(1, 2, 1 if label == '(g, 0)' else 2, projection='3d')
        b = qt.Bloch(fig=fig, axes=ax)

        b.add_states(state[0])
        
        for j in range(1, len(state), 1):
            b.add_line(state[j-1], state[j])
        
        b.add_states(state[-1])
        
        b.render()
        b.axes.set_title(f'Bloch sphere for {label} at tau = {tau_full[i]:.3f} s')

## Without approximation

Everything above lives in the **rotating frame**, where the drive is a static $A\sigma_x/2$ term. That picture is the result of two steps: moving to the frame rotating at $\omega_d$, and then discarding the counter-rotating terms (the RWA). The hardware delivers neither — it delivers a real voltage oscillating at 4.5 GHz.

So we now redo the whole sequence in the **lab frame**, with an explicitly time-dependent Hamiltonian and no approximation:

$$\mathcal{H}(t) = \underbrace{\frac{1}{2}\left(\omega_q + 2\chi n\right)\sigma_z}_{\texttt{H\_0},\;\texttt{H\_1}} \;+\; \underbrace{A\cos(\omega_d t)\,\sigma_x}_{\texttt{[H\_d, modulation]}}$$

QuTiP takes this as a list, `H = [H_0, [H_d, modulation]]`, where `modulation` is the scalar function $\cos(\omega_d t)$ that the solver evaluates at every time step. Note that the $\sigma_z$ term now carries the **full** qubit frequency, not a detuning: there is no frame rotating along with it any more.

**Why the pulse duration is unchanged.** Writing $\sigma_x = \sigma_+ + \sigma_-$ and dropping the counter-rotating half gives

$$A\cos(\omega_d t)\,\sigma_x \;\longrightarrow\; \frac{A}{2}\,\sigma_x$$

which is exactly the term used in the rotating-frame section. So `A * sigmax()` here and `A * sigmax()/2` there describe the *same* pulse, and $t_{\pi/2} = \pi/2A$ still lands on the equator — the printout below confirms $P_g = P_e = 0.500$.

**What is genuinely new:**

- **Micromotion.** The populations now carry a small ripple at $2\omega_d$ during the pulses — the counter-rotating term the RWA throws away. Zoom into the first plot to see it riding on the smooth Rabi curve.
- **Bloch–Siegert shift.** That same term pushes the qubit frequency by roughly $A^2/4\omega_q \approx 2\pi \times 0.56$ MHz, so a drive placed exactly at $\omega_q$ is in fact slightly detuned. Against $2\chi = 2\pi \times 13.3$ MHz that is a few percent — small here, but it is the reason real experiments *calibrate* the drive frequency instead of computing it.
- **Phase bookkeeping.** In the rotating frame the pulses have no clock. In the lab frame they do, and getting that clock right is the subject of the cell further down.

In [ ]:
# hamiltonian of qubit with a resonator driven

omega_s = 6.0 * 2 * np.pi  # qubit frequency
omega_q = 4.5 * 2 * np.pi  # resonator frequency
omega_d = omega_q  # drive frequency

g = 0.1 * 2 * np.pi  # coupling strength

chi_01 = g**2 / (omega_s - omega_q)  # dispersive shift

A = 0.1 * 2 * np.pi  # drive amplitude

N_photons = 5  # number of photons in the resonator

a = qt.destroy(N_photons)  # annihilation operator for the resonator
a_dag = a.dag()  # creation operator for the resonator

# hamiltonian of the system in the dispersive regime with a drive on the qubit
H_0 = 0.5 * omega_q * qt.sigmaz()
H_1 = 0.5 * (omega_q + 2 * chi_01) * qt.sigmaz()

H_d = A * qt.sigmax()  # drive Hamiltonian
modulation = lambda t, args: np.cos(args['omega_d'] * t)  # modulation function for the drive

psi_0 = qt.basis(2, 0)  # initial state of the qubit (ground state)
omega_d = omega_q  # drive frequency

In [ ]:
t_pi_2 = np.pi / (2 * A)  # time for a pi/2 rotation

t_limits = [0, t_pi_2]  # time limits for the simulation
n_steps = 1000  # number of time steps for the simulation
t_axis = np.linspace(*t_limits, n_steps)  # time axis for the simulation

H = [H_0, [H_d, modulation]]
args = {'omega_d': omega_d}

states_1 = qt.mesolve(H, psi_0, t_axis, args=args).states

In [ ]:
Pg = qt.expect(qt.projection(2, 0, 0), states_1)  # probability of being in the ground state of the qubit and vacuum of the resonator
Pe = qt.expect(qt.projection(2, 1, 1), states_1)  # probability of being in the excited state of the qubit and vacuum of the resonator


fig, ax = plt.subplots(1, 1, figsize=(12, 8), sharex=True)
ax.plot(t_axis, Pg, label='P(g, 0)', color='blue')
ax.plot(t_axis, Pe, label='P(e, 0)', color='orange')
ax.set_ylabel('Probability')
ax.legend()  

print(f"Time for a pi/2 rotation: {t_pi_2:.3f} s")
print(f"Probability of being in the ground state at t = {t_pi_2:.3f} s: {Pg[-1]:.3f}")
print(f"Probability of being in the excited state at t = {t_pi_2:.3f} s: {Pe[-1]:.3f}")

**Free evolution, lab frame.** `H_0` and `H_1` are now the *bare* qubit Hamiltonians, $\tfrac{1}{2}\omega_q\sigma_z$ and $\tfrac{1}{2}(\omega_q + 2\chi)\sigma_z$. During $\tau$ the Bloch vector spins at 4.5 GHz in **both** cases — thousands of radians over a single $\tau$. Nothing "sits still" any more; that was an artefact of the rotating frame.

The photon signal is now the tiny *difference* between two very fast rotations: $2\chi\tau$ riding on top of $\omega_q\tau$.

The solver still starts at $t = 0$ here, which is harmless — with the drive off the Hamiltonian is time-independent, so the propagator depends only on the *duration* $\tau$, not on when it starts. That stops being true the moment the drive comes back on.

In [ ]:
tau_pi = np.pi / (2 * chi_01) 

step = tau_pi / 100
tau_axis = np.arange(0, 3*tau_pi, step)

n_points = 500

states_tau_0 = []
states_tau_1 = []

psi_tau_0 = states_1[-1]  # state after the pi/2 rotation
psi_tau_1 = states_1[-1]  # state after the pi/2 rotation

for i, tau in enumerate(tau_axis):
    states_tau_0.append(qt.mesolve(H_0, psi_tau_0, np.linspace(0, tau, n_points)).states[-1])
    states_tau_1.append(qt.mesolve(H_1, psi_tau_1, np.linspace(0, tau, n_points)).states[-1])

### The clock matters: why `t_start` is not zero

This is the cell where the lab frame earns its keep. The rotating-frame version integrated the second pulse over `np.linspace(0, t_pi_2, ...)`. Here it is

```python
t_start = t_pi_2 + tau
time_points = np.linspace(t_start, t_start + t_pi_2, n_points_final)
```

Because `modulation` evaluates $\cos(\omega_d t)$ at **absolute** time, the second pulse does not restart its cosine from zero — it picks the oscillation up wherever it has got to. That is the simulation's version of a physical fact: in the lab the microwave generator is **never switched off**. It free-runs at GHz, and a fast RF switch (or an IQ mixer driven to zero envelope) simply blocks the path to the fridge between pulses. The generator keeps accumulating phase in the background, and that uninterrupted oscillation is the phase reference of the entire experiment.

**Two clocks run during $\tau$:**

$$\theta_\text{qubit}(\tau) = \left(\omega_q + 2\chi n\right)\tau, \qquad \theta_\text{drive}(\tau) = \omega_d\,\tau$$

The qubit feels the second pulse relative to the drive's phase, so what survives is the difference

$$\Delta\phi = \theta_\text{qubit} - \theta_\text{drive} = \left[\left(\omega_q - \omega_d\right) + 2\chi n\right]\tau \quad\xrightarrow{\;\omega_d\,=\,\omega_q\;}\quad 2\chi n\,\tau$$

which is precisely the $\varphi$ of the rotating-frame derivation. The GHz terms cancel identically and only the MHz signal is left. **The rotating frame is not merely a convenient change of variables — it is the statement that the generator is the clock against which the qubit is measured.**

**Try breaking it.** Replace the two lines above with `np.linspace(0, t_pi_2, n_points_final)` and re-run. The second pulse's rotation axis then sits at an angle $\omega_d\left(t_{\pi/2} + \tau\right)$ instead of along $x$; with $\omega_d \approx 28$ rad/ns and a $\tau$ step of $0.375$ ns, that axis swings by $\sim 10$ rad between neighbouring points and the fringe collapses into aliased noise. The same thing happens physically if the generator is switched off and restarted with a random phase between the two pulses: averaging over thousands of shots washes the interference out to a flat $P_e = 0.5$.

> 📎 **What actually has to be stable** is only the phase *between the two pulses of a single shot*. A random offset $\alpha$ common to **both** pulses is harmless: it rotates both pulse axes about $z$ by the same angle, and since the free evolution is itself a $z$ rotation, everything commutes,
>
> $$R_z(\alpha)\left[R_x(\pi/2)\,R_z(\varphi)\,R_x(\pi/2)\right]R_z(-\alpha)$$
>
> and the outer rotations do not touch the populations measured along $z$. This is why a lab can free-run its local oscillator with asynchronous triggering and still see clean fringes — and why *virtual Z gates* work at all: a $z$ rotation implemented purely by advancing the phase of every later pulse in software. In the paper the coherence only has to survive the $t_p = 380$ ns of one parity measurement, against $T_2 = 61\;\mu$s.

In [ ]:
states_final_0 = []
states_final_1 = []

save_full_0 = []
save_full_1 = []

n_points_final = 500

tau_pi = np.pi / (2 * chi_01) 
tau_pi_2 = tau_pi / 2
tau_pi_4 = tau_pi / 4

tau_full = [tau_pi_4, tau_pi_2, tau_pi]

for i, tau in enumerate(tau_axis):
    
    t_start = t_pi_2 + tau  # start time for the final pi/2 rotation
    t_end = t_start + t_pi_2  # end time for the final pi/2 rotation
    
    time_points = np.linspace(t_start, t_end, n_points_final)
    
    states_0 = qt.mesolve(H, states_tau_0[i], time_points, args={'omega_d': omega_d}).states
    states_1 = qt.mesolve(H, states_tau_1[i], time_points, args={'omega_d': omega_d}).states
    
    if np.isclose(tau, tau_pi_4) or np.isclose(tau, tau_pi_2) or np.isclose(tau, tau_pi):
        print(f"Saving full states for tau = {tau:.3f} s")
        save_full_0.append(states_0)
        save_full_1.append(states_1)
    
    states_final_0.append(states_0[-1])
    states_final_1.append(states_1[-1])

**Analytic check.** `pgnd` and `pexc` are the closed-form results derived in section 3,

$$P(g, n) = \sin^2\!\left(\frac{\delta\tau}{2}\right), \qquad P(e, n) = \cos^2\!\left(\frac{\delta\tau}{2}\right), \qquad \delta = 2\chi n$$

written with $\delta = 2\chi$ so that $\delta\tau/2 = \chi\tau$ — the same expression as the boxed one above, in the variable the code uses. They are overlaid as dashed black curves on the $n = 1$ panel of the next plot.

If the full time-dependent simulation lies on top of those curves, then the RWA, the rotating frame and the phase bookkeeping of the previous cell are all doing what we think they are. Any residual disagreement should show up as a **small loss of contrast or a constant offset**, not as a drift growing with $\tau$: the free evolution is exact here, so imperfections can only come from the pulses (Bloch–Siegert detuning making them slightly off-resonant).

In [ ]:
pgnd = lambda t, arg: np.sin(arg['delta'] * t /2)**2
pexc = lambda t, arg: np.cos(arg['delta'] * t /2)**2

In [ ]:
Pg0 = qt.expect(qt.projection(2, 0, 0), states_final_0)  # probability of being in the ground state of the qubit and vacuum of the resonator
Pe0 = qt.expect(qt.projection(2, 1, 1), states_final_0)  # probability of being in the excited state of the qubit and vacuum of the resonator
Pg1 = qt.expect(qt.projection(2, 0, 0), states_final_1)  # probability of being in the ground state of the qubit and vacuum of the resonator
Pe1 = qt.expect(qt.projection(2, 1, 1), states_final_1)  # probability of being in the excited state of the qubit and vacuum of the resonator

start, end = 0, 1000

delta = 2 * chi_01

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax[0].plot(tau_axis[start:end], Pg0[start:end], label='P(g, 0)', color='blue')
ax[0].plot(tau_axis[start:end], Pe0[start:end], label='P(e, 0)', color='orange')
ax[0].set_ylabel('Probability')
ax[0].legend()
ax[1].plot(tau_axis[start:end], Pg1[start:end], label='P(g, 1)', color='blue', lw=2)
ax[1].plot(tau_axis[start:end], pgnd(tau_axis[start:end], {'delta': delta}), label='P(g, 1) (analytical)', color='black', linestyle='--', lw=2, alpha=0.7)

ax[1].plot(tau_axis[start:end], Pe1[start:end], label='P(e, 1)', color='orange', lw=2)
ax[1].plot(tau_axis[start:end], pexc(tau_axis[start:end], {'delta': delta}), label='P(e, 1) (analytical)', color='black', linestyle='--', lw=2, alpha=0.7)
ax[1].axvline(tau_pi, color='red', linestyle='--', label=r'$\tau_\pi$')
ax[1].set_ylabel('Probability')
ax[1].set_xlabel('Free evolution time (ns)')
ax[1].legend()

### Where this leaves us

Both frames give the same fringe, so the rotating-frame model is trustworthy — and much cheaper, since it does not force the solver to resolve 4.5 GHz oscillations over a 100 ns window.

What is still missing, relative to the actual experiment:

- **Decoherence.** No collapse operators anywhere: $T_1$ and $T_2$ are infinite here. In the paper $T_1^q = 108\;\mu$s, $T_2^q = 61\;\mu$s, and dephasing during the parity measurement is an error channel indistinguishable from qubit decay.
- **A real cavity.** `a` is defined but never used; $n$ enters as a label on two Hamiltonians rather than as a Hilbert space. Enough for $n \in \{0, 1\}$, not enough for a coherent state or for measurement back-action.
- **Readout and repetition.** The paper's sensitivity comes from repeating this QND parity measurement many times and feeding the outcomes to a hidden Markov model, which is what suppresses false positives from qubit heating and decay.